In [15]:
!pip install openai

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.1 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 4.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------------- ------------------- 1.0/2.0 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 5.7 MB/s  0:00:00

   ----- ---------------------------------- 1/8 [sniffio]
   ---------- ----------------------------- 2/8 [pydantic-core]
   -------------------- ------------------- 4/8 [distro]
   -------------------- ------------------- 4/8 [distro]
   ------------------------------ --------- 6/8 [pydantic]
   ------------------------------ --------- 6/8 [pydantic]
   ------------------------------ --------- 6/8 [pydantic]
   ------------------------------ --------- 6/8 [pydantic]
   ------------------------------ --------- 6/8 [pydantic]
   -----------------------------

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="sk-proj-***********************************s1XItuwA")

Step 2 — System Prompt

In [34]:
system_prompt = """
You are an AI assistant that answers questions using GitHub documentation.

Always use the search tool to find relevant information before answering.

If search results are not sufficient, say you don't have enough information.
"""

STEP 3 — TOOL DEFINITION

In [35]:
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search in GitHub documentation",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query"
            }
        },
        "required": ["query"]
    }
}

STEP 4 — Agent

In [ ]:
question = "What is machine learning?"

chat_messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": question}
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages
)

print(response.output[0].content[0].text)

STEP 5 — Capture the tool call

In [ ]:
import json

call = response.output[0]

arguments = json.loads(call.arguments)

result = text_search(**arguments)

STEP 6 — Return it to the LLM

In [ ]:
call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result)
}

chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)

STEP 7 — Fixing the text_search function

In [ ]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Search GitHub documentation chunks.
    """
    return index.search(query, num_results=5)

STEP 8 — Creating an Agent

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    name="github_doc_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)

STEP 9 — Run

In [ ]:
result = await agent.run(
    user_prompt="What is machine learning?"
)

print(result.output)